In [ ]:
import re

POS_DICT = {

    # Pronouns
    "म": ("PRON", 0.99), "हामी": ("PRON", 0.99),
    "तिमी": ("PRON", 0.98), "तपाईं": ("PRON", 0.98),
    "उ": ("PRON", 0.97), "उनी": ("PRON", 0.97),

    # Determiners
    "यो": ("DET", 0.96), "त्यो": ("DET", 0.96),

    # Proper nouns
    "राम": ("PROPN", 0.99), "सीता": ("PROPN", 0.99),
    "नेपाल": ("PROPN", 0.99), "भारत": ("PROPN", 0.99),
    "काठमाडौं": ("PROPN", 0.99), "पोखरा": ("PROPN", 0.99),

    # Nouns
    "किताब": ("NOUN", 0.98), "घर": ("NOUN", 0.98),
    "विद्यालय": ("NOUN", 0.98), "देश": ("NOUN", 0.97),
    "खाना": ("NOUN", 0.97), "शहर": ("NOUN", 0.97),

    # Adjectives
    "राम्रो": ("ADJ", 0.98), "ठूलो": ("ADJ", 0.98),
    "सानो": ("ADJ", 0.97), "सुन्दर": ("ADJ", 0.97),

    # Verbs
    "गर्छ": ("VERB", 0.98), "गयो": ("VERB", 0.98),
    "पढ्छ": ("VERB", 0.98), "जान्छ": ("VERB", 0.97),
    "आयो": ("VERB", 0.97), "खान्छ": ("VERB", 0.97),

    # Auxiliary
    "छ": ("AUX", 0.99), "छन्": ("AUX", 0.99),
    "छु": ("AUX", 0.99),

    # Postpositions
    "मा": ("ADP", 0.99), "ले": ("ADP", 0.99),
    "लाई": ("ADP", 0.99), "बाट": ("ADP", 0.99),

    # Conjunction
    "र": ("CONJ", 0.99), "तर": ("CONJ", 0.99),

    # Adverbs
    "धेरै": ("ADV", 0.97), "अहिले": ("ADV", 0.97),

    # Numbers
    "एक": ("NUM", 0.98), "दुई": ("NUM", 0.98),
    "तीन": ("NUM", 0.98),
}


# TOKENIZER
def tokenize(text):
    text = re.sub(r'([।!?.,])', r' \1 ', text)
    return text.split()


# POS TAGGING
def get_tag(token):
    # punctuation
    if token in "।!?.,":
        return ("PUNCT", 1.0)

    # dictionary lookup
    if token in POS_DICT:
        return POS_DICT[token]

    # heuristic rules
    if token.endswith(("छ", "छन्", "छु")):
        return ("AUX", 0.6)
    elif token.endswith(("दै", "दा", "यो", "ए")):
        return ("VERB", 0.6)
    elif token.endswith(("ो", "ी", "ा")):
        return ("ADJ", 0.6)
    else:
        return ("NOUN", 0.6)


def pos_tag(tokens):
    tagged = []
    for tok in tokens:
        tag, score = get_tag(tok)
        tagged.append({
            "token": tok,
            "tag": tag,
            "score": score
        })
    return tagged


# DEPENDENCY PARSER
def find_root(tagged):
    for i, t in enumerate(tagged):
        if t["tag"] == "VERB":
            return i
    return 0


def assign_dependency(index, root_index, tag):
    if index == root_index:
        return "ROOT"
    elif tag in ("PRON", "PROPN") and index < root_index:
        return "nsubj"
    elif tag == "NOUN":
        return "obj"
    elif tag == "ADJ":
        return "amod"
    elif tag == "ADV":
        return "advmod"
    elif tag == "ADP":
        return "case"
    elif tag == "AUX":
        return "aux"
    else:
        return "dep"


def dependency_parse(tagged):
    root_index = find_root(tagged)

    deps = []
    for i, t in enumerate(tagged):
        dep = assign_dependency(i, root_index, t["tag"])
        deps.append({
            "token": t["token"],
            "dep": dep
        })
    return deps


# SENTENCE SPLIT
def split_sentences(text):
    return [s.strip() for s in re.split(r'[।!?]', text) if s.strip()]


# MAIN PARSER
def parse_text(text):
    sentences = split_sentences(text)
    results = []

    for sent in sentences:
        tokens = tokenize(sent)
        tagged = pos_tag(tokens)
        deps = dependency_parse(tagged)

        results.append({
            "sentence": sent,
            "tokens": tokens,
            "pos_tags": tagged,
            "dependencies": deps
        })

    return results


if __name__ == "__main__":

    while True:
        text = input("\nEnter Nepali sentence (or type 'exit'): ")

        if text.lower() == "exit":
            print("Exiting...")
            break

        result = parse_text(text)

        print("\n--- RESULT ---")
        for r in result:
            print("\nSentence:", r["sentence"])
            print("Tokens:", r["tokens"])

            print("POS Tags:")
            for t in r["pos_tags"]:
                print(f"  {t['token']} -> {t['tag']} ({t['score']})")

            print("Dependencies:")
            for d in r["dependencies"]:
                print(f"  {d['token']} -> {d['dep']}")


Enter Nepali sentence (or type 'exit'): राम किताब पढ्छ।

--- RESULT ---

Sentence: राम किताब पढ्छ
Tokens: ['राम', 'किताब', 'पढ्छ']
POS Tags:
  राम -> PROPN (0.99)
  किताब -> NOUN (0.98)
  पढ्छ -> VERB (0.98)
Dependencies:
  राम -> nsubj
  किताब -> obj
  पढ्छ -> ROOT
